# Lab 11: When Motor Commands Meet Muscle 
**Computational Sensorimotor Control**

This lab compares three muscle-controller combinations on the same reaching task:
1. **R-C Ca²⁺ λ** — R-C commands on the Gribble muscle model (no tendon)
2. **R-C Hill** — same R-C commands on Hill-type muscles (CE + SE + PE)
3. **KMHM** — R-C base + CE velocity reference feedback on Hill muscles

We run each model at two speeds (slow and fast), examine velocity profiles,
test equifinality under perturbation, and explore arc reaching with via-points.

---
## Part 0: Setup

In [ ]:
# Install the smc package (run once)
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.figsize': (10, 5), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'
})

from smc import (
    Arm, Q_REF, make_muscles, make_hill_muscles,
    lambda_for_posture, make_ramp,
    simulate_lambda, simulate_hill, simulate_kmhm,
)

arm = Arm()
fk = arm.forward_kinematics
ik = arm.inverse_kinematics
start_hand = np.array(fk(Q_REF))
dt = 0.0001

# Colors for consistent plotting
C_MJ   = '#27AE60'  # min-jerk (green)
C_CA   = '#2E86AB'  # R-C Ca²⁺ λ (blue)
C_HILL = '#E74C3C'  # R-C Hill (red)
C_KMHM = '#8E44AD'  # KMHM (purple)

print(f"Start: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm")
print(f"Q_REF: ({np.degrees(Q_REF[0]):.1f}°, {np.degrees(Q_REF[1]):.1f}°)")

---
## Part 1: Define the reaching target

Compute the target 12 cm to the right of the start position, and find the
λ values at start and target using `lambda_for_posture` from `smc`.

In [ ]:
# Target: 12 cm to the right
target_xy = start_hand + np.array([0.12, 0.0])
q_target = ik(target_xy[0], target_xy[1])

# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Compute λ values at start and target postures
# ════════════════════════════════════════════════════════════════
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

print(f"Target: ({target_xy[0]*100:.1f}, {target_xy[1]*100:.1f}) cm")
print(f"Δq: ({np.degrees(q_target[0]-Q_REF[0]):.1f}°, {np.degrees(q_target[1]-Q_REF[1]):.1f}°)")
print(f"λ_init: {li}")
print(f"λ_final: {lf}")

---
## Part 2: Slow Reach (500 ms λ ramp)

Create a 500 ms λ ramp using `make_ramp` and simulate all three models.
At this speed, the tendon is not stressed — all models should behave similarly.
*This reproduces Lecture Figure 3.*

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Create a slow λ ramp (500 ms duration, starting at 50 ms)
# ════════════════════════════════════════════════════════════════
RAMP_SLOW = 0.50
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

# Simulate all three models
T_SIM = 2.0
tg_s, _, hg_s, _ = simulate_lambda(lam_slow, T=T_SIM)
trh_s, _, hrh_s, _ = simulate_hill(lam_slow, T=T_SIM)
tk_s, _, hk_s, _ = simulate_kmhm(lam_slow, q_target, T=T_SIM, ramp_dur=RAMP_SLOW)

# Min-jerk benchmark (600 ms duration)
MJ_SLOW = 0.60
t_mj = np.arange(0, T_SIM, dt)
h_mj_slow = np.zeros((len(t_mj), 2))
for i, t in enumerate(t_mj):
    if t < 0.05:
        qt = Q_REF
    elif t < 0.05 + MJ_SLOW:
        tau = (t - 0.05) / MJ_SLOW
        s = 10*tau**3 - 15*tau**4 + 6*tau**5
        qt = Q_REF + s * (q_target - Q_REF)
    else:
        qt = q_target
    h_mj_slow[i] = fk(qt)

print("Slow reach simulations complete.")

In [ ]:
# ── Figure 1: Slow reach ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]; ax.set_title('A: Hand paths (slow reach)', fontweight='bold')
for h, c, l in [(h_mj_slow, C_MJ, 'Min-jerk'),
                 (hg_s, C_CA, 'R-C Ca²⁺ λ'),
                 (hrh_s, C_HILL, 'R-C Hill'),
                 (hk_s, C_KMHM, 'KMHM')]:
    ax.plot(h[:,0]*100, h[:,1]*100, color=c, lw=2, label=l)
ax.plot(*start_hand*100, 'ko', ms=8); ax.plot(*target_xy*100, 'k*', ms=12)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.legend(fontsize=8); ax.set_aspect('equal')

ax = axes[1]; ax.set_title('B: Displacement (slow reach)', fontweight='bold')
for t, h, c, l in [(t_mj, h_mj_slow, C_MJ, 'Min-jerk'),
                    (tg_s, hg_s, C_CA, 'R-C Ca²⁺ λ'),
                    (trh_s, hrh_s, C_HILL, 'R-C Hill'),
                    (tk_s, hk_s, C_KMHM, 'KMHM')]:
    ax.plot(t*1000, np.linalg.norm(h - start_hand, axis=1)*100, color=c, lw=2, label=l)
ax.axhline(12, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.legend(fontsize=8); ax.set_xlim(0, 1500)
plt.tight_layout(); plt.show()

---
## Part 3: Fast Reach (150 ms λ ramp)

Shorten the ramp to 150 ms. The SE stores elastic energy and releases it rapidly —
watch the overshoot in R-C Hill.
*This reproduces Lecture Figure 4.*

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Create a fast λ ramp (150 ms duration)
# ════════════════════════════════════════════════════════════════
RAMP_FAST = 0.15
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

# Simulate all three models
tg_f, _, hg_f, _ = simulate_lambda(lam_fast, T=T_SIM)
trh_f, _, hrh_f, _ = simulate_hill(lam_fast, T=T_SIM)
tk_f, _, hk_f, _ = simulate_kmhm(lam_fast, q_target, T=T_SIM, ramp_dur=RAMP_FAST)

# Min-jerk benchmark (300 ms duration)
MJ_FAST = 0.30
h_mj_fast = np.zeros((len(t_mj), 2))
for i, t in enumerate(t_mj):
    if t < 0.05:
        qt = Q_REF
    elif t < 0.05 + MJ_FAST:
        tau = (t - 0.05) / MJ_FAST
        s = 10*tau**3 - 15*tau**4 + 6*tau**5
        qt = Q_REF + s * (q_target - Q_REF)
    else:
        qt = q_target
    h_mj_fast[i] = fk(qt)

print("Fast reach simulations complete.")

In [ ]:
# ── Figure 2: Fast reach ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]; ax.set_title('A: Hand paths (fast reach)', fontweight='bold')
for h, c, l in [(h_mj_fast, C_MJ, 'Min-jerk'),
                 (hg_f, C_CA, 'R-C Ca²⁺ λ'),
                 (hrh_f, C_HILL, 'R-C Hill'),
                 (hk_f, C_KMHM, 'KMHM')]:
    ax.plot(h[:,0]*100, h[:,1]*100, color=c, lw=2, label=l)
ax.plot(*start_hand*100, 'ko', ms=8); ax.plot(*target_xy*100, 'k*', ms=12)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.legend(fontsize=8); ax.set_aspect('equal')

ax = axes[1]; ax.set_title('B: Displacement (fast reach)', fontweight='bold')
for t, h, c, l in [(t_mj, h_mj_fast, C_MJ, 'Min-jerk'),
                    (tg_f, hg_f, C_CA, 'R-C Ca²⁺ λ'),
                    (trh_f, hrh_f, C_HILL, 'R-C Hill'),
                    (tk_f, hk_f, C_KMHM, 'KMHM')]:
    ax.plot(t*1000, np.linalg.norm(h - start_hand, axis=1)*100, color=c, lw=2, label=l)
ax.axhline(12, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.legend(fontsize=8); ax.set_xlim(0, 1200)
plt.tight_layout(); plt.show()

---
## Part 4: Compute Metrics

Write a function that computes peak velocity, movement time (5%–95%),
endpoint error, and overshoot from a hand trajectory.

In [ ]:
def compute_metrics(t, h, target):
    v = np.linalg.norm(np.diff(h, axis=0) / dt, axis=1) * 100
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    fd = d[-1]

    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN: Compute peak velocity and overshoot percentage
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════

    err = np.linalg.norm(h[-1] - target) * 100
    # Movement time: 5% to 95% of final displacement
    t5 = t95 = 0
    for k in range(len(d)):
        if t5 == 0 and d[k] >= 0.05 * fd: t5 = t[k]
        if t95 == 0 and d[k] >= 0.95 * fd: t95 = t[k]; break
    tmov = (t95 - t5) * 1000
    # Asymmetry: time to peak velocity relative to movement window
    above = np.where(v > 0.05 * pv)[0]
    asym = (np.argmax(v) - above[0]) / (above[-1] - above[0]) if len(above) > 1 else 0.5
    return {'pv': pv, 'tmov': tmov, 'err': err, 'ovsh': ovsh, 'asym': asym}

In [ ]:
# Print comparison table
print(f"{'':12s} {'Speed':5s} {'PeakV':>8s} {'tMov':>7s} {'Error':>7s} {'Ovsh':>6s} {'Asym':>5s}")
print("-" * 56)
for speed, datasets in [
    ('SLOW', [(t_mj,h_mj_slow,'Min-jerk'), (tg_s,hg_s,'Ca²⁺ λ'),
              (trh_s,hrh_s,'R-C Hill'), (tk_s,hk_s,'KMHM')]),
    ('FAST', [(t_mj,h_mj_fast,'Min-jerk'), (tg_f,hg_f,'Ca²⁺ λ'),
              (trh_f,hrh_f,'R-C Hill'), (tk_f,hk_f,'KMHM')])]:
    for t, h, name in datasets:
        m = compute_metrics(t, h, target_xy)
        print(f"{name:12s} {speed:5s} {m['pv']:6.0f}cm/s {m['tmov']:5.0f}ms "
              f"{m['err']:5.2f}cm {m['ovsh']:5.1f}% {m['asym']:5.2f}")

---
## Part 5: Velocity Profiles

The velocity asymmetry reveals the muscle mechanics:
- **R-C Ca²⁺ λ** peaks late → calcium filter delay
- **R-C Hill / KMHM** peak early → SE catapult releases stored energy
- **Min-jerk** peaks at 50% (symmetric by construction)
*This reproduces Lecture Figure 5.*

In [ ]:
# ── Figure 3: Velocity profiles ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]; ax.set_title('A: Hand speed — slow reach', fontweight='bold')
for t, h, c, l in [(t_mj, h_mj_slow, C_MJ, 'Min-jerk'),
                    (tg_s, hg_s, C_CA, 'R-C Ca²⁺ λ'),
                    (trh_s, hrh_s, C_HILL, 'R-C Hill'),
                    (tk_s, hk_s, C_KMHM, 'KMHM')]:
    v = np.linalg.norm(np.diff(h, axis=0) / dt, axis=1) * 100
    ax.plot(t[:-1]*1000, v, color=c, lw=2, label=l)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Speed (cm/s)')
ax.legend(fontsize=8); ax.set_xlim(0, 1500)

ax = axes[1]; ax.set_title('B: Hand speed — fast reach', fontweight='bold')
for t, h, c, l in [(t_mj, h_mj_fast, C_MJ, 'Min-jerk'),
                    (tg_f, hg_f, C_CA, 'R-C Ca²⁺ λ'),
                    (trh_f, hrh_f, C_HILL, 'R-C Hill'),
                    (tk_f, hk_f, C_KMHM, 'KMHM')]:
    v = np.linalg.norm(np.diff(h, axis=0) / dt, axis=1) * 100
    ax.plot(t[:-1]*1000, v, color=c, lw=2, label=l)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Speed (cm/s)')
ax.legend(fontsize=8); ax.set_xlim(0, 1200)
plt.tight_layout(); plt.show()

---
## Part 6: Equifinality — Perturbation Test

Apply a 5 N·m torque perturbation (50 ms) at 300 ms during the slow reach.
Does the hand return to the same endpoint? How does the recovery differ?
*This reproduces Lecture Figure 6.*

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Define a perturbation function that applies [5, -3] N·m
#   torque between 300 and 350 ms, and zero otherwise
# ════════════════════════════════════════════════════════════════
def pert_fn(t):
    ...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

# Perturbed simulations (slow ramp)
_, _, hg_p, _ = simulate_lambda(lam_slow, T=T_SIM, perturb_fn=pert_fn)
_, _, hrh_p, _ = simulate_hill(lam_slow, T=T_SIM, perturb_fn=pert_fn)
_, _, hk_p, _ = simulate_kmhm(lam_slow, q_target, T=T_SIM, ramp_dur=RAMP_SLOW, perturb_fn=pert_fn)

print("Perturbed simulations complete.")
print(f"Endpoint diff — R-C Ca²⁺ λ: {np.linalg.norm(hg_s[-1]-hg_p[-1])*100:.4f} cm")
print(f"Endpoint diff — R-C Hill:    {np.linalg.norm(hrh_s[-1]-hrh_p[-1])*100:.4f} cm")
print(f"Endpoint diff — KMHM:        {np.linalg.norm(hk_s[-1]-hk_p[-1])*100:.4f} cm")

In [ ]:
# ── Figure 4: Equifinality test ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]; ax.set_title('A: Hand paths — perturbation test', fontweight='bold')
for h, hp, c, l in [(hg_s, hg_p, C_CA, 'R-C Ca²⁺ λ'),
                     (hrh_s, hrh_p, C_HILL, 'R-C Hill'),
                     (hk_s, hk_p, C_KMHM, 'KMHM')]:
    ax.plot(h[:,0]*100, h[:,1]*100, color=c, lw=2, label=l)
    ax.plot(hp[:,0]*100, hp[:,1]*100, color=c, lw=2, ls='--', alpha=0.6)
ax.plot(*start_hand*100, 'ko', ms=8); ax.plot(*target_xy*100, 'k*', ms=12)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.legend(fontsize=8); ax.set_aspect('equal')
ax.text(0.02, 0.02, 'Dashed = perturbed', transform=ax.transAxes, fontsize=8, style='italic', color='gray')

ax = axes[1]; ax.set_title('B: Perturbation recovery', fontweight='bold')
for t, h, hp, c, l in [(tg_s, hg_s, hg_p, C_CA, 'R-C Ca²⁺ λ'),
                        (trh_s, hrh_s, hrh_p, C_HILL, 'R-C Hill'),
                        (tk_s, hk_s, hk_p, C_KMHM, 'KMHM')]:
    diff = np.linalg.norm(h - hp, axis=1) * 100
    ax.plot(t*1000, diff, color=c, lw=2, label=l)
ax.axvline(300, color='gray', ls=':', lw=0.8); ax.axvline(350, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Dist. from unperturbed (cm)')
ax.legend(fontsize=8); ax.set_xlim(0, 2000)
plt.tight_layout(); plt.show()

---
## Part 7: Can Velocity Feedback Fix the Oscillations?

KMHM adds CE velocity feedback with a 25 ms delay. What if the delay were
shorter? Test with 1 ms delay (physiologically impossible, but informative).
*This reproduces Lecture Figure 7.*

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Simulate KMHM with 1 ms delay (both unperturbed and perturbed)
# ════════════════════════════════════════════════════════════════
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# ════════════════════════════════════════════════════════════════

print("KMHM 1ms delay simulations complete.")

In [ ]:
# ── Figure 5: Delay comparison (3 panels) ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel A: Displacement unperturbed
ax = axes[0]; ax.set_title('A: Displacement (unperturbed)', fontweight='bold')
for t, h, c, l, ls_ in [(tg_s, hg_s, C_CA, 'R-C Ca²⁺ λ', '-'),
                         (trh_s, hrh_s, C_HILL, 'R-C Hill', '-'),
                         (tk_s, hk_s, C_KMHM, 'KMHM (25 ms)', '-'),
                         (tk_s, hk1_u, '#27AE60', 'KMHM (1 ms)', '--')]:
    ax.plot(t*1000, np.linalg.norm(h - start_hand, axis=1)*100, color=c, lw=2, ls=ls_, label=l)
ax.axhline(12, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.legend(fontsize=7); ax.set_xlim(0, 2000)

# Panel B: Displacement perturbed
ax = axes[1]; ax.set_title('B: Displacement (perturbed)', fontweight='bold')
for t, h, c, l, ls_ in [(tg_s, hg_p, C_CA, 'R-C Ca²⁺ λ', '-'),
                         (trh_s, hrh_p, C_HILL, 'R-C Hill', '-'),
                         (tk_s, hk_p, C_KMHM, 'KMHM (25 ms)', '-'),
                         (tk_s, hk1_p, '#27AE60', 'KMHM (1 ms)', '--')]:
    ax.plot(t*1000, np.linalg.norm(h - start_hand, axis=1)*100, color=c, lw=2, ls=ls_, label=l)
ax.axhline(12, color='gray', ls=':', lw=1)
ax.axvline(300, color='gray', ls=':', lw=0.8); ax.axvline(350, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.legend(fontsize=7); ax.set_xlim(0, 2000)

# Panel C: Recovery
ax = axes[2]; ax.set_title('C: Perturbation recovery', fontweight='bold')
for t, h, hp, c, l, ls_ in [
    (tg_s, hg_s, hg_p, C_CA, 'R-C Ca²⁺ λ', '-'),
    (trh_s, hrh_s, hrh_p, C_HILL, 'R-C Hill', '-'),
    (tk_s, hk_s, hk_p, C_KMHM, 'KMHM (25 ms)', '-'),
    (tk_s, hk1_u, hk1_p, '#27AE60', 'KMHM (1 ms)', '--')]:
    ax.plot(t*1000, np.linalg.norm(h - hp, axis=1)*100, color=c, lw=2, ls=ls_, label=l)
ax.axvline(300, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Dist. from unperturbed (cm)')
ax.legend(fontsize=7); ax.set_xlim(0, 2000)
plt.tight_layout(); plt.show()

print("Key finding: reducing delay from 25 ms to 1 ms barely changes the oscillations.")
print("The oscillations are intrinsic to the CE+SE dynamics, not the feedback delay.")

---
## Part 8: Arc Reaching with Via-Points

The arc from Weeks 8–9: a 6 cm semicircle from start to 12 cm right.
EPH controls *where* the arm goes, not *how* — so we need via-points
along the arc to make the hand curve upward.
*This reproduces Lecture Figure 8.*

In [ ]:
# Arc geometry
R_ARC = 0.06
arc_center = start_hand + np.array([R_ARC, 0])

def dense_arc(R, n=500):
    """Dense points along the desired arc for deviation computation."""
    c = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([c[0] + R*np.cos(th), c[1] + R*np.sin(th)])

def arc_viapoints(R, N):
    """N equally spaced via-points along the arc."""
    c = start_hand + np.array([R, 0])
    angles = np.linspace(np.pi, 0, N)
    return np.column_stack([c[0] + R*np.cos(angles), c[1] + R*np.sin(angles)])

arc_path = dense_arc(R_ARC)
print(f"Arc: R = {R_ARC*100:.0f} cm, peak height = {(arc_center[1]+R_ARC)*100:.1f} cm")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Build a λ ramp through 6 via-points (compute λ at each
#   via-point using lambda_for_posture, then interpolate between them)
# ════════════════════════════════════════════════════════════════
N_VIA = 6
vps = arc_viapoints(R_ARC, N_VIA)
seg_dur = 0.80 / (N_VIA - 1)  # 160 ms per segment

...  # YOUR CODE HERE
...  # YOUR CODE HERE

def lam_vp(t):
    t_start = 0.05
    if t < t_start: return lam_vias[0].copy()
    elapsed = t - t_start
    si = int(elapsed / seg_dur)
    if si >= N_VIA - 1: return lam_vias[-1].copy()
    f = (elapsed - si * seg_dur) / seg_dur
    return lam_vias[si] + f * (lam_vias[si+1] - lam_vias[si])
# ════════════════════════════════════════════════════════════════

print(f"{N_VIA} via-points, {seg_dur*1000:.0f} ms per segment")

In [ ]:
# Simulate arc reaching
tga, _, hga, _ = simulate_lambda(lam_vp, T=T_SIM)
trha, _, hrha, _ = simulate_hill(lam_vp, T=T_SIM)
tka, _, hka, _ = simulate_kmhm(lam_vp, q_target, T=T_SIM, ramp_dur=0.80)

# Min-jerk arc (prescribed path along semicircle)
h_mj_arc = np.zeros((len(t_mj), 2))
for i, t in enumerate(t_mj):
    if t < 0.05: s = 0.0
    elif t < 0.05 + 0.80:
        tau = (t - 0.05) / 0.80; s = 10*tau**3 - 15*tau**4 + 6*tau**5
    else: s = 1.0
    th = np.pi * (1 - s)
    h_mj_arc[i] = [arc_center[0] + R_ARC*np.cos(th), arc_center[1] + R_ARC*np.sin(th)]

def path_deviation(hand, arc):
    return np.array([np.min(np.linalg.norm(arc - p, axis=1)) for p in hand]) * 100

print("Arc simulations complete.")

In [ ]:
# ── Figure 6: Arc reaching ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]; ax.set_title('A: Arc reaching (6 via-points)', fontweight='bold')
ax.plot(arc_path[:,0]*100, arc_path[:,1]*100, '--', color='gray', lw=1.5, label='Desired arc')
ax.plot(h_mj_arc[:,0]*100, h_mj_arc[:,1]*100, color=C_MJ, lw=2, label='Min-jerk')
ax.plot(hga[:,0]*100, hga[:,1]*100, color=C_CA, lw=2, label='R-C Ca²⁺ λ')
ax.plot(hrha[:,0]*100, hrha[:,1]*100, color=C_HILL, lw=2, label='R-C Hill')
ax.plot(hka[:,0]*100, hka[:,1]*100, color=C_KMHM, lw=2, label='KMHM')
ax.plot(vps[:,0]*100, vps[:,1]*100, 'D', color='#F39C12', ms=7, zorder=5, label='Via-points')
ax.plot(*start_hand*100, 'ko', ms=8, zorder=5)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_aspect('equal')
ax.legend(fontsize=7, loc='upper left'); ax.set_xlim(-12, 5); ax.set_ylim(59, 72)

ax = axes[1]; ax.set_title('B: Arc deviation', fontweight='bold')
ax.plot(t_mj*1000, path_deviation(h_mj_arc, arc_path), color=C_MJ, lw=2, label='Min-jerk')
ax.plot(tga*1000, path_deviation(hga, arc_path), color=C_CA, lw=2, label='R-C Ca²⁺ λ')
ax.plot(trha*1000, path_deviation(hrha, arc_path), color=C_HILL, lw=2, label='R-C Hill')
ax.plot(tka*1000, path_deviation(hka, arc_path), color=C_KMHM, lw=2, label='KMHM')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Distance from arc (cm)')
ax.legend(fontsize=8); ax.set_xlim(0, 1500)
plt.tight_layout(); plt.show()

print(f"Max arc deviation — R-C Ca²⁺ λ: {np.max(path_deviation(hga, arc_path)):.2f} cm")
print(f"Max arc deviation — R-C Hill:    {np.max(path_deviation(hrha, arc_path)):.2f} cm")
print(f"Max arc deviation — KMHM:        {np.max(path_deviation(hka, arc_path)):.2f} cm")

---
## Summary

| Feature | R-C Ca²⁺ λ | R-C Hill | KMHM |
|---|---|---|---|
| Muscle model | No tendon | CE + SE + PE | CE + SE + PE |
| Control | R-C λ ramps | Same λ → STIM | Same + v_CE feedback |
| Slow overshoot | ~1% | ~2% | ~1% |
| Fast overshoot | ~4% | ~13% | ~8% |
| Perturbation recovery | Smooth | Oscillatory | Oscillatory |
| Equifinality | Exact | Approximate | Approximate |

**Key takeaways:**
1. At slow speeds, all three models perform identically — the tendon doesn't matter
2. At fast speeds, the SE catapult boosts peak velocity 60% but causes 13% overshoot
3. KMHM's velocity feedback reduces overshoot to 8% but cannot eliminate oscillations — even with 1 ms delay
4. The oscillations are intrinsic to the CE+SE spring-mass dynamics
5. Via-points enable curved-path following, but EPH needs external specification of the path
6. **Week 12** introduces OFC: optimal feedback gains that account for the full plant dynamics